# DHEIS (District Health and Education Intelligence System)


## STEP 1: Connect to Supabase via JDBC

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import mlflow

jdbc_url = "jdbc:postgresql://aws-1-ap-south-1.pooler.supabase.com:5432/postgres?sslmode=require"

connection_properties = {
    "user": "postgres.panzknkodnmiwobvxrci",
    "password": dbutils.secrets.get(scope="capstone-secrets", key="supabase-password"),
    "driver": "org.postgresql.Driver"
}

In [0]:
pw = dbutils.secrets.get(scope="capstone-secrets", key="supabase-password")
print(len(pw) > 0)   # should print True

True


## STEP 2: Read the three core tables from Supabase

In [0]:
df_district = spark.read.jdbc(url=jdbc_url, table="dim_district", properties=connection_properties)
df_health = spark.read.jdbc(url=jdbc_url, table="raw_health", properties=connection_properties)
df_education = spark.read.jdbc(url=jdbc_url, table="raw_education", properties=connection_properties)

{"ts": "2026-09-06 19:45:10.025", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjQ5NjY2MzAzNzEyNTAyMBABIAEyJDAxYTA3ODQwLTYwMGUtN2I2Ni1hMmE1LWFmZDdhNDU2MmViZDokMTNjYzVjZmQtNjA0MS0zNTliLWJkYjMtYTYwNTFjZTE0ZGVlSgwIp4X31AYQgLP/xANQAVgBYAFoteHkkJHFow0=.", "context": {}}
{"ts": "2026-09-06 19:45:10.025", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjQ5NjY2MzAzNzEyNTAyMBABIAEyJDAxYTA3ODQwLTYwMGUtN2I2Ni1hMmE1LWFmZDdhNDU2MmViZDokMTNjYzVjZmQtNjA0MS0zNTliLWJkYjMtYTYwNTFjZTE0ZGVlSgwIp4X31AYQgLP/xANQAVgBYAFoteHkkJHFow0=.", "context": {}}
{"ts": "2026-09-06 19:45:10.025", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjQ5NjY2MzAzNzEyNTAyMBABIAEyJDAxYTA3ODQwLTYwMGUtN2I2Ni1hMmE1LWFmZDdhNDU2MmViZDokMTNjYzVjZmQtNjA0MS0zNTliLWJkYjMtYTYwNTFjZTE0ZGVl

Check the row counts came through correctly.

In [0]:
# Verify row counts from bronze tables (already loaded from Supabase)
district_count = spark.table("bronze_dim_district").count()
health_count = spark.table("bronze_raw_health").count()
education_count = spark.table("bronze_raw_education").count()

print("dim_district rows:", district_count, "(expect 707)")
print("raw_health rows:", health_count, "(expect 706)")
print("raw_education rows:", education_count, "(expect 603)")

dim_district rows: 707 (expect 707)
raw_health rows: 706 (expect 706)
raw_education rows: 603 (expect 603)


In [0]:
print("dim_district rows:", df_district.count())     # expect 707
print("raw_health rows:", df_health.count())          # expect 706
print("raw_education rows:", df_education.count())    # expect 603

dim_district rows: 707
raw_health rows: 706
raw_education rows: 603


## STEP 3: Write BRONZE Delta tables (raw copy, as-is from Postgres)

In [0]:
df_district.write.format("delta").mode("overwrite").saveAsTable("bronze_dim_district")
df_health.write.format("delta").mode("overwrite").saveAsTable("bronze_raw_health")
df_education.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_raw_education")

## STEP 4: Build the SILVER layer — join health + education via `district_id`
**What each Tier 1 column means:**

| Column | Meaning |
|---|---|
| `institutional_births_pct` | % of births in a hospital/clinic, not at home |
| `full_immunization_pct` | % of children fully vaccinated |
| `stunted_pct` / `wasted_pct` / `underweight_pct` | % of children malnourished (long-term / acute / general) |
| `child_anaemia_pct` | % of children with low blood iron |
| `household_sanitation_pct` | % of population with proper toilet access |
| `health_insurance_pct` | % of households with any health insurance |
| `overall_literacy` | % of population that can read and write |
| `dropout_rate_pct` | % of enrolled students who dropped out |
| `primary_pupil_teacher_ratio` | Students per teacher, primary level |
| `pct_schools_electricity` / `_water` / `_road_connected` | % of schools with that facility |

In [0]:
silver_df = (
    df_district
    .join(df_health, on="district_id", how="inner")
    .join(df_education, on="district_id", how="inner")
)

print("Silver (joined) row count:", silver_df.count())   # expect 602

Silver (joined) row count: 602


In [0]:
silver_df = silver_df.withColumn(
    "true_pupil_teacher_ratio",
    F.col("total_enrollment") / F.col("total_teachers")
)

## STEP 5: Write the SILVER Delta table

In [0]:
# Drop duplicate state_name columns by converting to pandas and back
# (Spark Connect doesn't easily handle duplicate column names)
silver_df_pandas = silver_df.toPandas()

# Remove duplicate columns, keeping first occurrence
silver_df_pandas = silver_df_pandas.loc[:, ~silver_df_pandas.columns.duplicated()]

# Convert back to Spark DataFrame
silver_df_clean = spark.createDataFrame(silver_df_pandas)

silver_df_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_district_combined")


In [0]:
from pyspark.sql.functions import round
from pyspark.sql.types import DoubleType, FloatType, DecimalType

# Identify all floating-point columns
float_cols = [f.name for f in silver_df_clean.schema.fields if isinstance(f.dataType, (DoubleType, FloatType, DecimalType))]

# Round each float column to 2 decimal places
for col_name in float_cols:
    silver_df_clean = silver_df_clean.withColumn(col_name, round(col_name, 2))



In [0]:
display(silver_df_clean.limit(5))

district_id,district_name,state_name,institutional_births_pct,full_immunization_pct,stunted_pct,wasted_pct,underweight_pct,child_anaemia_pct,pregnant_women_anaemia_pct,diarrhoea_prevalence_pct,ari_prevalence_pct,antenatal_care_4visits_pct,postnatal_care_pct,household_electricity_pct,household_drinking_water_pct,household_sanitation_pct,health_insurance_pct,womens_literacy_health_survey,overall_literacy,female_literacy,male_literacy,total_enrollment,govt_enrollment,private_enrollment,school_age_population,primary_pupil_teacher_ratio,upper_primary_pupil_teacher_ratio,total_schools,schools_with_water,schools_with_electricity,schools_with_girls_toilet,schools_with_boys_toilet,schools_with_road_connectivity,pct_sc_population,pct_st_population,dropout_rate_pct,pct_govt_enrollment,pct_private_enrollment,pct_schools_road_connected,pct_schools_electricity,pct_schools_water,pct_schools_girls_toilet,girl_dropout_rate,boy_dropout_rate,dropout_gender_gap,literacy_gender_gap,toilet_gender_gap,total_teachers,true_pupil_teacher_ratio
94,PASHCHIM CHAMPARAN,BIHAR,79.50,62.10,43.50,13.20,30.30,61.60,61.40,22.10,6.60,25.40,57.30,93.50,99.70,43.30,20.00,52.50,58.06,46.79,68.16,919779,818120,66052,609570.00,2265.00,321.00,2925,2755,975,2560,2564,2718,14.08,6.35,0.71,88.90,7.20,92.90,33.30,94.20,87.50,0.61,0.81,-0.20,21.37,-4,16027,57.39
29,PAPUM PARE,ARUNACHAL PRADESH,88.00,60.70,29.70,9.00,15.60,53.40,18.60,4.60,1.90,39.30,53.20,98.70,92.90,75.70,26.90,78.20,82.14,76.65,87.33,48711,31718,16248,18589.00,36.00,3.00,421,298,195,416,410,227,0.00,66.38,1.44,65.10,33.40,53.90,46.30,70.80,98.80,1.46,1.42,0.04,10.68,6,3529,13.8
362,HINGOLI,MAHARASTRA,94.00,76.90,37.40,25.80,38.90,72.50,53.30,10.10,3.20,66.60,82.40,98.10,92.40,69.00,14.70,76.50,76.04,64.73,86.73,187660,111834,75814,122653.00,164.00,55.00,1182,1182,895,1170,1151,1151,15.51,9.51,1.86,59.60,40.40,97.40,75.70,100.00,99.00,1.59,2.12,-0.53,22.00,19,7501,25.02
638,JHANSI,UTTAR PRADESH,92.90,44.50,40.90,25.20,39.30,70.30,35.70,4.90,3.40,36.60,80.40,95.20,96.80,73.20,11.40,72.50,76.37,64.88,86.58,326477,161107,162078,197752.00,618.00,147.00,2577,2512,1548,2512,2477,2489,28.15,0.19,0.35,49.30,49.60,96.60,60.10,97.50,97.50,0.32,0.37,-0.05,21.70,35,13677,23.87
284,ERNAKULAM,KERALA,99.10,82.60,22.00,17.10,19.40,36.40,null,1.80,2.40,82.20,96.10,100.00,98.20,99.30,46.20,99.30,95.68,94.27,97.14,358890,42712,302237,219910.00,110.00,57.00,1364,1364,1345,1348,1341,1343,8.18,0.50,2.55,11.90,84.20,98.50,98.60,100.00,98.80,2.02,3.06,-1.04,2.87,7,23508,15.27


## STEP 6: Quick sanity check — real district data looks right

In [0]:
display(silver_df_clean.select("district_name", "state_name", "overall_literacy",
                          "institutional_births_pct", "dropout_rate_pct").limit(15))

district_name,state_name,overall_literacy,institutional_births_pct,dropout_rate_pct
PASHCHIM CHAMPARAN,BIHAR,58.06,79.50,0.71
PAPUM PARE,ARUNACHAL PRADESH,82.14,88.00,1.44
HINGOLI,MAHARASTRA,76.04,94.00,1.86
JHANSI,UTTAR PRADESH,76.37,92.90,0.35
ERNAKULAM,KERALA,95.68,99.10,2.55
MADHEPURA,BIHAR,53.78,75.00,1.31
SIDDHARTHNAGAR,UTTAR PRADESH,61.81,69.70,1.48
CHANDIGARH,CHANDIGARH,86.43,96.90,2.48
BIKANER,RAJASTHAN,65.92,90.00,1.06
RAJNANDGAON,CHHATTISGARH,76.97,95.50,1.32


## STEP 7: First real EDA — summary statistics

In [0]:
silver_df_clean.select(
    "overall_literacy", "dropout_rate_pct", "institutional_births_pct",
    "full_immunization_pct", "stunted_pct", "household_sanitation_pct"
).describe().display()

summary,overall_literacy,dropout_rate_pct,institutional_births_pct,full_immunization_pct,stunted_pct,household_sanitation_pct
count,571,602,602,594,602,602
mean,73.051664,1.370698,88.333555,77.523232,33.606811,71.746844
stddev,10.016293981399699,0.7902720807998685,12.169209268919944,12.209101420415656,8.498660173978326,14.497453962215559
min,37.22,0.22,21.40,38.30,13.80,29.20
max,98.76,6.38,100.00,100.00,60.60,99.90


## STEP 8: Correlation check — does health relate to education in this data?

In [0]:
corr_pairs = [
    ("dropout_rate_pct", "overall_literacy"),
    ("dropout_rate_pct", "institutional_births_pct"),
    ("stunted_pct", "overall_literacy"),
    ("literacy_gender_gap", "dropout_gender_gap"),
]
for col1, col2 in corr_pairs:
    corr_val = silver_df_clean.stat.corr(col1, col2)
    print(f"Correlation ({col1}, {col2}): {__builtins__.round(corr_val, 3)}")

Correlation (dropout_rate_pct, overall_literacy): 0.158
Correlation (dropout_rate_pct, institutional_births_pct): 0.152
Correlation (stunted_pct, overall_literacy): -0.274
Correlation (literacy_gender_gap, dropout_gender_gap): 0.109


**Result:**     
 stunted vs literacy = -0.274 (moderate).     
 dropout vs literacy = 0.158,     
 dropout vs institutional births = 0.152 (both weak).      
 Literacy gender gap vs dropout gender gap: 0.109 (weak positive)   
No single indicator strongly predicts another — exactly why a combined index needs multiple indicators.

## STEP 9: Missing value check on the silver table

In [0]:
check_cols = ["overall_literacy", "female_literacy", "school_age_population",
              "full_immunization_pct", "pregnant_women_anaemia_pct"]
              
silver_df_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in check_cols
]).display()

overall_literacy,female_literacy,school_age_population,full_immunization_pct,pregnant_women_anaemia_pct
31,28,29,8,105


# PRIORITY INDEX BUILD — DHEIS

## STEP 10: Define which indicators are "good" vs "bad"
"Good" = higher raw value means LESS vulnerable (e.g. literacy). "Bad" = higher raw value means MORE vulnerable (e.g. dropout rate). This matters because we need to flip the "good" ones before combining everything onto one common vulnerability scale.

In [0]:
good_health_indicators = ["institutional_births_pct", "full_immunization_pct",
                           "household_sanitation_pct", "health_insurance_pct"]
bad_health_indicators = ["stunted_pct", "wasted_pct", "underweight_pct", "child_anaemia_pct"]
good_edu_indicators = ["overall_literacy", "pct_schools_electricity",
                        "pct_schools_water", "pct_schools_road_connected"]
bad_edu_indicators = ["dropout_rate_pct", "true_pupil_teacher_ratio"]  

all_good = good_health_indicators + good_edu_indicators
all_bad = bad_health_indicators + bad_edu_indicators
all_indicators = all_good + all_bad

## STEP 11: Percentile-rank normalize every indicator to 0-100


In [0]:
df = silver_df_clean

for c in all_indicators:
    if c in all_good:
        window = Window.orderBy(F.col(c).asc())   # lowest raw value = most vulnerable
    else:
        window = Window.orderBy(F.col(c).desc())  # highest raw value = most vulnerable
    df = df.withColumn(f"{c}_vuln", F.percent_rank().over(window) * 100)

## STEP 12: Build the health and education sub-scores
Each sub-score is simply the average vulnerability across its own indicators — 0 means least vulnerable, 100 means most vulnerable.

In [0]:
health_vuln_cols = [f"{c}_vuln" for c in good_health_indicators + bad_health_indicators]
edu_vuln_cols = [f"{c}_vuln" for c in good_edu_indicators + bad_edu_indicators]

df = df.withColumn("health_subscore", sum(F.col(c) for c in health_vuln_cols) / len(health_vuln_cols))
df = df.withColumn("education_subscore", sum(F.col(c) for c in edu_vuln_cols) / len(edu_vuln_cols))

## STEP 13: Combine into one Priority Index

In [0]:
df = df.withColumn("priority_index", (F.col("health_subscore") + F.col("education_subscore")) / 2)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## STEP 14: Bucket districts into High / Medium / Low tiers
Using percentiles, not fixed numbers — top 20% most vulnerable = High, bottom 20% = Low, everyone else = Medium.

In [0]:
quantiles = df.approxQuantile("priority_index", [0.80, 0.20], 0.01)
high_cutoff, low_cutoff = quantiles[0], quantiles[1]

df = df.withColumn(
    "priority_tier",
    F.when(F.col("priority_index") >= high_cutoff, "High")
     .when(F.col("priority_index") <= low_cutoff, "Low")
     .otherwise("Medium")
)

print(f"High priority cutoff (index >=): {high_cutoff:.2f}")
print(f"Low priority cutoff (index <=): {low_cutoff:.2f}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


High priority cutoff (index >=): 63.16
Low priority cutoff (index <=): 35.87


Check the tier counts.

In [0]:
df.groupBy("priority_tier").count().display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


priority_tier,count
Low,118
Medium,359
High,125


## STEP 15: Verify the normalization is working correctly
`avg_gap` should be small for High, Medium, AND Low — a big, near-constant gap across all three would mean something is scaled unevenly, not that a real pattern exists.

In [0]:
df.groupBy("priority_tier").agg(
    F.avg("health_subscore").alias("avg_health"),
    F.avg("education_subscore").alias("avg_education"),
    F.round(F.avg("health_subscore") - F.avg("education_subscore"), 2).alias("avg_gap")
).orderBy("priority_tier").display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


priority_tier,avg_health,avg_education,avg_gap
High,72.27537437603995,67.07953410981699,5.2
Low,27.307947206633013,31.14912058057663,-3.84
Medium,49.41207551017572,48.42826177973263,0.98


**Real result:** gaps of roughly 4.6 / 1.4 / -4.3 across High/Medium/Low — small and tier-varying, confirming the normalization is fair.

## STEP 16: Classify each district's dominant driver
Is a district's vulnerability mainly coming from health, education, or both equally? This matters for real-world action — health-driven needs mobile health camps, education-driven needs teacher deployment.

In [0]:
df = df.withColumn(
    "dominant_driver",
    F.when((F.col("health_subscore") - F.col("education_subscore")) > 5, "Health-driven")
     .when((F.col("health_subscore") - F.col("education_subscore")) < -5, "Education-driven")
     .otherwise("Both equally")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Check the breakdown, by tier.

In [0]:
print("Dominant driver breakdown, BY TIER:")
df.groupBy("priority_tier", "dominant_driver").count() \
    .orderBy("priority_tier", F.desc("count")).display()

Dominant driver breakdown, BY TIER:


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


priority_tier,dominant_driver,count
High,Health-driven,65
High,Both equally,34
High,Education-driven,26
Low,Education-driven,56
Low,Health-driven,33
Low,Both equally,29
Medium,Health-driven,152
Medium,Education-driven,122
Medium,Both equally,85


Check the overall breakdown, across all 602 districts.

In [0]:
print("\nOverall dominant driver breakdown, all 602 districts:")
df.groupBy("dominant_driver").count().orderBy(F.desc("count")).display()


Overall dominant driver breakdown, all 602 districts:


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dominant_driver,count
Health-driven,250
Education-driven,204
Both equally,148


## STEP 17: State-level analysis
Which states have the most High Priority districts — both as a raw count and as a percentage of that state's total districts.

In [0]:
state_summary = df.groupBy("state_name").agg(
    F.count("*").alias("total_districts"),
    F.sum(F.when(F.col("priority_tier") == "High", 1).otherwise(0)).alias("high_priority_count")
).withColumn(
    "pct_high_priority", F.round(F.col("high_priority_count") / F.col("total_districts") * 100, 1)
).orderBy(F.desc("pct_high_priority"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Display it, filtered to states with at least 3 districts (avoids tiny-sample noise).

In [0]:
print("States ranked by % of their districts that are High Priority :")
state_summary.filter(F.col("total_districts") >= 3).display()

States ranked by % of their districts that are High Priority :


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


state_name,total_districts,high_priority_count,pct_high_priority
PUDUCHERRY,3,3,100.0
TAMIL NADU,29,27,93.1
KERALA,14,13,92.9
HARYANA,21,17,81.0
MIZORAM,7,5,71.4
PUNJAB,19,13,68.4
HIMACHAL PRADESH,11,7,63.6
UTTARAKHAND,13,8,61.5
KARNATAKA,27,10,37.0
ANDHRA PRADESH,13,2,15.4


## STEP 18: Load Tier 2 detail tables from Supabase
Tier 2 is extra detail used only for the Power BI diagnostic drill-down page — never in the model or the priority index.

**What each Tier 2 column means:**

| Column | Meaning |
|---|---|
| `repeater_rate_pct` | % of students who repeated a grade rather than dropping out or progressing |
| `trained_teachers_total` / `total_teachers` | Trained teachers out of total teachers |
| `schools_with_playground` / `_boundary_wall` / `_computer` | Count of schools with that facility |
| `vaccine_bcg_pct` / `_polio_pct` / `_dpt_pct` / `_measles_pct` / `_rotavirus_pct` / `_hepatitis_b_pct` | % of children vaccinated with that specific vaccine |

In [0]:
df_education_detail = spark.read.jdbc(url=jdbc_url, table="raw_education_detail", properties=connection_properties)
df_health_detail = spark.read.jdbc(url=jdbc_url, table="raw_health_detail", properties=connection_properties)

## STEP 19: Write Tier 2 BRONZE tables

In [0]:
df_education_detail.write.format("delta").mode("overwrite").saveAsTable("bronze_raw_education_detail")
df_health_detail.write.format("delta").mode("overwrite").saveAsTable("bronze_raw_health_detail")

## STEP 20: Build the Tier 2 SILVER table

In [0]:
silver_district_detail = (
    df_district
    .join(df_education_detail, on="district_id", how="left")
    .join(df_health_detail, on="district_id", how="left")
)
silver_district_detail.write.format("delta").mode("overwrite").saveAsTable("silver_district_detail")

print("silver_district_detail row count:", silver_district_detail.count())

silver_district_detail row count: 707


## STEP 21: Tier 2 EDA — missing values

In [0]:
tier2_cols = [
    "repeater_rate_pct", "trained_teachers_total", "total_teachers",
    "schools_with_playground", "schools_with_boundary_wall", "schools_with_computer",
    "vaccine_bcg_pct", "vaccine_polio_pct", "vaccine_dpt_pct",
    "vaccine_measles_pct", "vaccine_rotavirus_pct", "vaccine_hepatitis_b_pct"
]

silver_district_detail.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in tier2_cols
]).display()

repeater_rate_pct,trained_teachers_total,total_teachers,schools_with_playground,schools_with_boundary_wall,schools_with_computer,vaccine_bcg_pct,vaccine_polio_pct,vaccine_dpt_pct,vaccine_measles_pct,vaccine_rotavirus_pct,vaccine_hepatitis_b_pct
104,104,104,104,104,104,14,14,14,14,14,14


## STEP 22: Tier 2 EDA — summary statistics: repeater rate and teacher training

In [0]:
silver_district_detail.select("repeater_rate_pct", "trained_teachers_total", "total_teachers").describe().display()

summary,repeater_rate_pct,trained_teachers_total,total_teachers
count,603,603,603
mean,0.5206799336650082918740,1762.8291873963515,11925.08623548922
stddev,0.8130309369046882,2117.0958989912147,8119.048346267523
min,0.000000000000000000,0,151
max,5.770000000000000000,15741,71009


## STEP 23: Tier 2 EDA — summary statistics: individual vaccines

In [0]:
vaccine_cols = [c for c in tier2_cols if c.startswith("vaccine_")]
silver_district_detail.select(*vaccine_cols).describe().display()

summary,vaccine_bcg_pct,vaccine_polio_pct,vaccine_dpt_pct,vaccine_measles_pct,vaccine_rotavirus_pct,vaccine_hepatitis_b_pct
count,693,693,693,693,693,693
mean,95.0111111111111111111111,81.5020202020202020202020,87.3409812409812409812410,88.4561327561327561327561,39.1310245310245310245310,84.6562770562770562770563
stddev,4.594559556788168,10.546163122507053,8.451047078536245,8.219848250060746,32.18137415409616,9.928804740567104
min,65.900000000000000000,47.600000000000000000,53.300000000000000000,53.200000000000000000,0.000000000000000000,39.500000000000000000
max,100.000000000000000000,100.000000000000000000,100.000000000000000000,100.000000000000000000,100.000000000000000000,100.000000000000000000


## STEP 24: Which vaccine lags the most, nationally?
Useful for the AI brief later — "this district is behind specifically on X vaccine," not just a generic immunization number.

In [0]:
avg_by_vaccine = silver_district_detail.select(
    *[F.avg(c).alias(c) for c in vaccine_cols]
).collect()[0].asDict()

for vaccine, avg_val in sorted(avg_by_vaccine.items(), key=lambda x: x[1]):
    print(f"{vaccine}: {__builtins__.round(avg_val, 1)}%")

vaccine_rotavirus_pct: 39.1%
vaccine_polio_pct: 81.5%
vaccine_hepatitis_b_pct: 84.7%
vaccine_dpt_pct: 87.3%
vaccine_measles_pct: 88.5%
vaccine_bcg_pct: 95.0%


## STEP 25: Does repeater rate tell a different story than dropout rate?
A district could "hide" its real problem by keeping struggling students enrolled but not progressing, rather than dropping out — this wouldn't show up in `dropout_rate_pct` alone.

In [0]:
joined = silver_district_detail.join(
    df.select("district_id", "dropout_rate_pct", "priority_tier", "priority_index"),
    on="district_id", how="inner"
)

corr_repeater_dropout = joined.stat.corr("repeater_rate_pct", "dropout_rate_pct")
print(f"Correlation (repeater_rate_pct, dropout_rate_pct): {__builtins__.round(corr_repeater_dropout, 3)}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Correlation (repeater_rate_pct, dropout_rate_pct): -0.135


## STEP 26: Does vaccine coverage actually differ by priority tier?
A trust check — High Priority districts (based on Tier 1 data) SHOULD also show lower vaccine coverage in this independent Tier 2 data, if the priority index is capturing something real.

In [0]:
joined_with_avg_vaccine = joined.withColumn(
    "avg_vaccine_coverage",
    sum(F.col(c) for c in vaccine_cols) / len(vaccine_cols)
)

joined_with_avg_vaccine.groupBy("priority_tier").agg(
    F.avg("avg_vaccine_coverage").alias("avg_vaccine_coverage"),
    F.count("*").alias("num_districts")
).orderBy("priority_tier").display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


priority_tier,avg_vaccine_coverage,num_districts
High,85.2475206611570165,125
Low,75.5798022598870169,118
Medium,78.6362910798121972,359


## STEP 27: Facility availability summary

In [0]:
silver_district_detail.select("schools_with_playground", "schools_with_boundary_wall", "schools_with_computer").describe().display()

summary,schools_with_playground,schools_with_boundary_wall,schools_with_computer
count,603,603,603
mean,1318.5107794361525,1388.5721393034826,562.1326699834162
stddev,985.134674402872,1022.7167370647279,581.1177472681724
min,3,20,8
max,5862,8153,5474


## STEP 28: Write the GOLD table (Tier 1 only)
**Design decision:** Tier 2 stays in its own separate table (`silver_district_detail`), joined in Power BI only on the diagnostic page. The gold table stays focused on what the model and priority index actually use.

In [0]:
gold_cols = ["district_id", "district_name", "state_name","health_subscore", "education_subscore",  
             "priority_index", "priority_tier", "dominant_driver"] + all_indicators + \
            ["literacy_gender_gap", "dropout_gender_gap", "toilet_gender_gap",
             "pct_sc_population", "pct_st_population","primary_pupil_teacher_ratio"]

gold_df = df.select(*gold_cols)

gold_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_district_priority")

print("Gold table written. Row count:", gold_df.count())


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold table written. Row count: 602


---
# ML MODEL SECTION

## STEP 29: Define the target and features
**Target:** `priority_tier`. **Features:** the 14 raw Tier 1 indicators only — `priority_index`, `health_subscore`, `education_subscore`, and `dominant_driver` are deliberately excluded, since they're calculated FROM these same features and would let the model "cheat" instead of learning a real pattern.

In [0]:
pdf = gold_df.toPandas()

feature_cols = [
    "institutional_births_pct", "full_immunization_pct", "household_sanitation_pct", "health_insurance_pct",
    "stunted_pct", "wasted_pct", "underweight_pct", "child_anaemia_pct",
    "overall_literacy", "pct_schools_electricity", "pct_schools_water", "pct_schools_road_connected",
    "dropout_rate_pct", "true_pupil_teacher_ratio"
]
target_col = "priority_tier"

X = pdf[feature_cols].copy()
y = pdf[target_col].copy()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Check the feature shape and target balance.

In [0]:
print("Feature matrix shape:", X.shape)
print("\nTarget distribution:")
print(y.value_counts())

Feature matrix shape: (602, 14)

Target distribution:
priority_tier
Medium    359
High      125
Low       118
Name: count, dtype: int64


Check for missing values (handled inside each model pipeline in STEP 32, not here).

In [0]:
print(X.isna().sum()[X.isna().sum() > 0])

full_immunization_pct     8
overall_literacy         31
dtype: int64


## STEP 30: Train / test split + stratified cross-validation

Hold out 20% of districts as an untouched final test set.
The remaining 80% is used for model training and 5-fold stratified
cross-validation for hyperparameter tuning.

The test set is not used during model selection and is evaluated only once
after the best model has been selected.

In [0]:
from sklearn.model_selection import train_test_split, StratifiedKFold

# Keep the full dataset for final production predictions
X_full = X.copy()
y_full = y.copy()

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    stratify=y_full,
    random_state=42
)

print("Full dataset:", X_full.shape)
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining tier distribution:")
print(y_train.value_counts())

print("\nTest tier distribution:")
print(y_test.value_counts())

# 5-fold CV ONLY inside the training set
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("\nTraining folds:")
for fold_num, (train_idx, val_idx) in enumerate(
    skf.split(X_train, y_train), 1
):
    fold_tiers = y_train.iloc[val_idx].value_counts().to_dict()

    print(
        f"Fold {fold_num}: "
        f"train={len(train_idx)}, "
        f"validation={len(val_idx)}, "
        f"validation tier balance={fold_tiers}"
    )

Full dataset: (602, 14)
Training set: (481, 14)
Test set: (121, 14)

Training tier distribution:
priority_tier
Medium    287
High      100
Low        94
Name: count, dtype: int64

Test tier distribution:
priority_tier
Medium    72
High      25
Low       24
Name: count, dtype: int64

Training folds:
Fold 1: train=384, validation=97, validation tier balance={'Medium': 58, 'High': 20, 'Low': 19}
Fold 2: train=385, validation=96, validation tier balance={'Medium': 58, 'High': 20, 'Low': 18}
Fold 3: train=385, validation=96, validation tier balance={'Medium': 57, 'High': 20, 'Low': 19}
Fold 4: train=385, validation=96, validation tier balance={'Medium': 57, 'High': 20, 'Low': 19}
Fold 5: train=385, validation=96, validation tier balance={'Medium': 57, 'High': 20, 'Low': 19}


## STEP 31: Baseline model
A simple "always guess the most common tier" model — every real model must beat this.

In [0]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, accuracy_score

baseline = DummyClassifier(strategy="most_frequent")
baseline_preds = cross_val_predict(baseline, X, y, cv=skf)

print("=== BASELINE (majority class) ===")
print("Accuracy:", __builtins__.round(accuracy_score(y, baseline_preds), 3))
print()
print(classification_report(y, baseline_preds))

=== BASELINE (majority class) ===
Accuracy: 0.596

              precision    recall  f1-score   support

        High       0.00      0.00      0.00       125
         Low       0.00      0.00      0.00       118
      Medium       0.60      1.00      0.75       359

    accuracy                           0.60       602
   macro avg       0.20      0.33      0.25       602
weighted avg       0.36      0.60      0.45       602



/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## STEP 32: Define three real models
Logistic Regression (simple, explainable), Random Forest (many trees voting), Gradient Boosting (trees correcting each other's mistakes). Missing-value imputation is built INSIDE each pipeline — this means the saved model can handle brand new future data on its own, and avoids a subtle leakage issue from imputing before the CV split.

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

mlflow.autolog()

models = {
    "logistic_regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42))
    ]),
    "gradient_boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=42))
    ]),
}

2026/09/06 19:48:55 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/09/06 19:48:55 WARNING mlflow.utils.autologging_utils: MLflow spark autologging is known to be compatible with 3.3.0 <= pyspark, but the installed version is 4.1.0. If you encounter errors during autologging, try upgrading / downgrading pyspark to a compatible version, or try upgrading MLflow.
2026/09/06 19:48:55 WARNING mlflow.tracking.fluent: Exception raised while enabling autologging for pyspark: MLflow Spark dataset autologging is not supported on Databricks shared clusters or Databricks serverless clusters.
2026/09/06 19:48:55 WARNING mlflow.utils.autologging_utils: MLflow pyspark.ml autologging is known to be compatible with 3.3.0 <= pyspark, but the installed version is 4.1.0. If you encounter errors during autologging, try upgrading / downgrading pyspark to a compatible version, or try upgrading MLflow.
2026/09/06 19:48:55 WARNING mlflow.tracking.fluent: Exception raised while e

## STEP 33: Train and compare all three models

In [0]:
results = {}

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        preds = cross_val_predict(model, X_train, y_train, cv=skf)
        acc = accuracy_score(y_train, preds)
        report = classification_report(y_train, preds, output_dict=True,zero_division=0)

        mlflow.log_metric("cv_accuracy", acc)
        mlflow.log_metric("high_tier_recall", report["High"]["recall"])
        mlflow.log_metric("high_tier_precision", report["High"]["precision"])

        results[name] = {"accuracy": acc, "report": report, "preds": preds}

        print(f"=== {name.upper()} ===")
        print("Training CV Accuracy:", __builtins__.round(acc, 3))
        print(classification_report(y_train, preds))
        print()

2026/09/06 19:48:57 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-bcc6d667647246ee92cbaf99153018bc?o=7474656442921141
2026/09/06 19:49:07 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-9c654bd15f034a28b741baa73f0fe661?o=7474656442921141
2026/09/06 19:49:13 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to 

=== LOGISTIC_REGRESSION ===
Training CV Accuracy: 0.923
              precision    recall  f1-score   support

        High       0.96      0.91      0.93       100
         Low       0.91      0.83      0.87        94
      Medium       0.92      0.96      0.94       287

    accuracy                           0.92       481
   macro avg       0.93      0.90      0.91       481
weighted avg       0.92      0.92      0.92       481




2026/09/06 19:49:31 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-51b2afa4fe1e4ba5a3b19d08415acfae?o=7474656442921141
2026/09/06 19:49:37 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-76aca01bc84f48478f37c1bdd95f0273?o=7474656442921141
2026/09/06 19:49:42 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to 

=== RANDOM_FOREST ===
Training CV Accuracy: 0.884
              precision    recall  f1-score   support

        High       0.93      0.80      0.86       100
         Low       0.94      0.72      0.82        94
      Medium       0.86      0.97      0.91       287

    accuracy                           0.88       481
   macro avg       0.91      0.83      0.86       481
weighted avg       0.89      0.88      0.88       481




2026/09/06 19:50:00 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-77e3495f1faf4e629d90b7bf22cee37a?o=7474656442921141
2026/09/06 19:50:06 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-815a5ce8e8cf413fbe1c3008c1dd78c5?o=7474656442921141
2026/09/06 19:50:13 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to 

=== GRADIENT_BOOSTING ===
Training CV Accuracy: 0.884
              precision    recall  f1-score   support

        High       0.88      0.84      0.86       100
         Low       0.91      0.77      0.83        94
      Medium       0.88      0.94      0.91       287

    accuracy                           0.88       481
   macro avg       0.89      0.85      0.87       481
weighted avg       0.88      0.88      0.88       481




## STEP 34: Light hyperparameter tuning
A small, bounded search over a few settings for each model, to check whether the fixed settings in STEP 32 were leaving performance on the table.

In [0]:
from sklearn.model_selection import GridSearchCV

param_grids = {
    "logistic_regression": {"clf__C": [0.1, 1.0, 10.0]},
    "random_forest": {"clf__n_estimators": [100, 200, 300], "clf__max_depth": [4, 6, 8]},
    "gradient_boosting": {"clf__n_estimators": [100, 150, 200], "clf__max_depth": [2, 3, 4]},
}

tuned_results = {}

Run the tuning search for each model.

In [0]:
from sklearn.metrics import make_scorer, recall_score

# Custom scorer for the High Priority class
def high_tier_recall(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        output_dict=True,
        zero_division=0
    )
    return report["High"]["recall"]

high_recall_scorer = make_scorer(high_tier_recall)

tuned_results = {}

for name, model in models.items():
    with mlflow.start_run(run_name=f"{name}_tuned"):

        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            cv=skf,
            scoring=high_recall_scorer,
            n_jobs=-1
        )

        grid.fit(X_train, y_train)

        tuned_preds = cross_val_predict(
        grid.best_estimator_,
        X_train,
        y_train,
        cv=skf
        )


        tuned_acc = accuracy_score(y_train, tuned_preds)

        tuned_report = classification_report(
            y_train,
            tuned_preds,
            output_dict=True,
            zero_division=0
        )

        high_recall = tuned_report["High"]["recall"]
        high_precision = tuned_report["High"]["precision"]
        high_f1 = tuned_report["High"]["f1-score"]

        mlflow.log_params(grid.best_params_)
        mlflow.log_metric("tuned_cv_accuracy", tuned_acc)
        mlflow.log_metric("tuned_high_recall", high_recall)
        mlflow.log_metric("tuned_high_precision", high_precision)
        mlflow.log_metric("tuned_high_f1", high_f1)

        tuned_results[name] = {
            "accuracy": tuned_acc,
            "report": tuned_report,
            "preds": tuned_preds,
            "best_params": grid.best_params_,
            "best_estimator": grid.best_estimator_
        }

        print(f"=== {name.upper()} TUNED ===")
        print("Best parameters:", grid.best_params_)
        print("Accuracy:", __builtins__.round(tuned_acc, 3))
        print("High-tier recall:", __builtins__.round(high_recall, 3))
        print("High-tier precision:", __builtins__.round(high_precision, 3))
        print("High-tier F1:", __builtins__.round(high_f1, 3))
        print()

2026/09/06 19:50:40 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-f33207fd1c76425c8f91e033a2257add?o=7474656442921141
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-b351095b4332440f836cb18eb7630b4a?o=7474656442921141
2026/09/06 19:50:48 INFO mlflow.sklearn.utils: Logging the 5 best runs, no runs will be omitted.
2026/09/06 19:50:49 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Lo

=== LOGISTIC_REGRESSION TUNED ===
Best parameters: {'clf__C': 10.0}
Accuracy: 0.925
High-tier recall: 0.94
High-tier precision: 0.931
High-tier F1: 0.935



2026/09/06 19:51:26 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-c3aa6189bfe54587b823d6e7f43c7b50?o=7474656442921141
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-47540f17dd344acb863fd9f1f357ba25?o=7474656442921141
2026/09/06 19:51:34 INFO mlflow.sklearn.utils: Logging the 5 best runs, 4 runs will be omitted.
2026/09/06 19:51:35 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Log

=== RANDOM_FOREST TUNED ===
Best parameters: {'clf__max_depth': 8, 'clf__n_estimators': 200}
Accuracy: 0.892
High-tier recall: 0.83
High-tier precision: 0.912
High-tier F1: 0.869



2026/09/06 19:52:32 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-0be4c444ca87437681c9ac487c2a4000?o=7474656442921141
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-f0872653147842f7b67a259f7a1988b4?o=7474656442921141
2026/09/06 19:52:40 INFO mlflow.sklearn.utils: Logging the 5 best runs, 4 runs will be omitted.
2026/09/06 19:52:42 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Log

=== GRADIENT_BOOSTING TUNED ===
Best parameters: {'clf__max_depth': 4, 'clf__n_estimators': 100}
Accuracy: 0.875
High-tier recall: 0.85
High-tier precision: 0.876
High-tier F1: 0.863



## STEP 35: Compare all untuned and tuned models, pick the winner
Ranked by **High-tier recall**, not overall accuracy — missing a genuinely High Priority district is more costly than a false alarm on a Medium one.

In [0]:
comparison = pd.DataFrame({
    f"{name}_untuned": {
        "accuracy": res["accuracy"], "high_precision": res["report"]["High"]["precision"],
        "high_recall": res["report"]["High"]["recall"], "high_f1": res["report"]["High"]["f1-score"],
    } for name, res in results.items()
} | {
    f"{name}_tuned": {
        "accuracy": res["accuracy"], "high_precision": res["report"]["High"]["precision"],
        "high_recall": res["report"]["High"]["recall"], "high_f1": res["report"]["High"]["f1-score"],
    } for name, res in tuned_results.items()
}).T.round(3).sort_values("high_recall", ascending=False)

display(comparison)

accuracy,high_precision,high_recall,high_f1
0.925,0.931,0.94,0.935
0.923,0.958,0.91,0.933
0.875,0.876,0.85,0.863
0.884,0.884,0.84,0.862
0.892,0.912,0.83,0.869
0.884,0.93,0.8,0.86


Pick the best model based on the comparison table.

In [0]:
best_run_name = comparison.index[0]
is_tuned = best_run_name.endswith("_tuned")
best_model_name = best_run_name.replace("_tuned", "").replace("_untuned", "")
best_model = tuned_results[best_model_name]["best_estimator"] if is_tuned else models[best_model_name]

print(f"Best model selected from training CV: {best_run_name}")
print(f"Model type: {best_model_name}")

Best model selected from training CV: logistic_regression_tuned
Model type: logistic_regression


## STEP 36: Final evaluation on the untouched test set

The selected model has never seen the test districts.
Fit it on the training data and evaluate once on the held-out test set.
These are the final model performance metrics.

In [0]:
# Fit the selected model ONLY on the training data
best_model.fit(X_train, y_train)

# Final prediction on untouched test data
test_preds = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_preds)
test_report = classification_report(
    y_test,
    test_preds,
    output_dict=True,
    zero_division=0
)

test_high_recall = test_report["High"]["recall"]
test_high_precision = test_report["High"]["precision"]
test_high_f1 = test_report["High"]["f1-score"]

print("=== FINAL TEST SET RESULTS ===")
print("Test Accuracy:", __builtins__.round(test_accuracy, 3))
print("High-tier Recall:", __builtins__.round(test_high_recall, 3))
print("High-tier Precision:", __builtins__.round(test_high_precision, 3))
print("High-tier F1:", __builtins__.round(test_high_f1, 3))

print("\nFull classification report:")
print(
    classification_report(
        y_test,
        test_preds,
        zero_division=0
    )
)

2026/09/06 19:53:15 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '08d62bad5d42416b8d73815e12ac9830', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/06 19:53:15 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-5b924c7e44cd40959fa7fffcc63422c5?o=7474656442921141


=== FINAL TEST SET RESULTS ===
Test Accuracy: 0.95
High-tier Recall: 0.96
High-tier Precision: 0.96
High-tier F1: 0.96

Full classification report:
              precision    recall  f1-score   support

        High       0.96      0.96      0.96        25
         Low       0.92      0.92      0.92        24
      Medium       0.96      0.96      0.96        72

    accuracy                           0.95       121
   macro avg       0.94      0.94      0.94       121
weighted avg       0.95      0.95      0.95       121



## STEP 37: Baseline, evaluated on the SAME held-out test set
This makes the "does my model beat the baseline" comparison fair — both the baseline and the final model are now scored on the identical 121 unseen districts, not different sample sizes as before.

In [0]:
# ---- Baseline, evaluated on the SAME held-out test set as the real model ----
baseline_final = DummyClassifier(strategy="most_frequent")
baseline_final.fit(X_train, y_train)

baseline_test_preds = baseline_final.predict(X_test)
baseline_test_accuracy = accuracy_score(y_test, baseline_test_preds)
baseline_test_report = classification_report(y_test, baseline_test_preds, output_dict=True, zero_division=0)

print("=== BASELINE — TEST SET RESULTS (same 121 districts as the final model) ===")
print("Test Accuracy:", __builtins__.round(baseline_test_accuracy, 3))
print("High-tier Recall:",  __builtins__.round(baseline_test_report["High"]["recall"], 3))
print()

print("=== HONEST COMPARISON: BASELINE vs FINAL MODEL, same test set ===")
comparison_final = pd.DataFrame({
    "baseline": {
        "accuracy": baseline_test_accuracy,
        "high_recall": baseline_test_report["High"]["recall"],
    },
    best_run_name: {
        "accuracy": test_accuracy,
        "high_recall": test_high_recall,
    }
}).T.round(3)

display(comparison_final)

2026/09/06 19:53:21 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '298c0eec44ca4d9ba5d0f0beb55a34ec', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/06 19:53:21 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-ed0c50be14844c0eb9b6ccaf63ce692a?o=7474656442921141
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric

=== BASELINE — TEST SET RESULTS (same 121 districts as the final model) ===
Test Accuracy: 0.595
High-tier Recall: 0.0

=== HONEST COMPARISON: BASELINE vs FINAL MODEL, same test set ===


accuracy,high_recall
0.595,0.0
0.95,0.96


On the same 121 held-out districts, a naive baseline achieves 59.5% accuracy but 0% recall on High Priority districts — meaning it never once identifies a district that actually needs urgent attention. My final model achieves 95% accuracy and 96% recall on High Priority districts, on data it never saw during training.

## STEP 38: Error analysis — which districts does the model get wrong?

In [0]:
# Build error-analysis dataframe ONLY for the untouched test set

test_indices = X_test.index

error_df = pdf.loc[
    test_indices,
    ["district_name", "state_name", "priority_tier"]
].copy()

error_df["predicted_tier"] = test_preds

error_df["correct"] = (
    error_df["priority_tier"]
    == error_df["predicted_tier"]
)

print(
    "Final test error rate:",
    __builtins__.round((~error_df["correct"]).mean(), 3)
)

Final test error rate: 0.05


Look specifically at High Priority districts the model MISSED — the costliest error type.

In [0]:
missed_high = error_df[(error_df["priority_tier"] == "High") & (error_df["predicted_tier"] != "High")]
print(f"High Priority districts the model MISSED: {len(missed_high)}")
display(missed_high[["district_name", "state_name", "predicted_tier"]])

High Priority districts the model MISSED: 1


district_name,state_name,predicted_tier
ALMORA,UTTARAKHAND,Medium


Check for any state-level pattern in the errors.

In [0]:
error_by_state = error_df.groupby("state_name")["correct"].agg(["mean", "count"]).sort_values("mean")
print("\nStates with lowest model accuracy :")
display(error_by_state[error_by_state["count"] >= 5].head(10))


States with lowest model accuracy :


mean,count
0.8,5
0.8333333333333334,12
0.9,10
1.0,11
1.0,6
1.0,5
1.0,6
1.0,5
1.0,7
1.0,9


## STEP 39: Fairness check
Tests whether model accuracy holds steady across gender-gap and SC/ST population groups. These columns are used ONLY here, never as model features.

In [0]:
fairness_df = pdf.loc[
    test_indices,
    [
        "district_name",
        "state_name",
        "literacy_gender_gap",
        "pct_sc_population",
        "pct_st_population",
        "priority_tier"
    ]
].copy()

fairness_df["predicted_tier"] = test_preds

fairness_df["correct"] = (
    fairness_df["priority_tier"]
    == fairness_df["predicted_tier"]
)

median_gap = fairness_df["literacy_gender_gap"].median()

fairness_df["gender_gap_group"] = np.where(
    fairness_df["literacy_gender_gap"] > median_gap,
    "High gap",
    "Low gap"
)

print("Final test-set accuracy by gender-gap group:")

display(
    fairness_df
    .groupby("gender_gap_group")["correct"]
    .agg(["mean", "count"])
)

Final test-set accuracy by gender-gap group:


mean,count
0.95,60
0.9508196721311475,61


In [0]:
fairness_df["sc_st_total"] = fairness_df["pct_sc_population"] + fairness_df["pct_st_population"]
median_sc_st = fairness_df["sc_st_total"].median()
fairness_df["sc_st_group"] = np.where(fairness_df["sc_st_total"] > median_sc_st, "High SC/ST %", "Low SC/ST %")

print("\nModel accuracy by SC/ST population group:")
display(fairness_df.groupby("sc_st_group")["correct"].agg(["mean", "count"]))


Model accuracy by SC/ST population group:


mean,count
0.9322033898305084,59
0.967741935483871,62


## STEP 40: Register the final model
Includes a model signature (required by Databricks Unity Catalog) describing exactly what input the model expects and what output it produces.

In [0]:
from mlflow.models import infer_signature

# After final test evaluation, retrain the selected model on all available districts for production use.
best_model.fit(X_full, y_full)

sample_predictions = best_model.predict(X_full.head(5))
signature = infer_signature(X_full, sample_predictions)

2026/09/06 19:57:43 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '8a629f2ec3d54aeeb14faf5614216465', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/06 19:57:43 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-5a3817f1da164b94acde713d5587840b?o=7474656442921141
2026/09/06 19:57:49 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.


In [0]:
# Convert Decimal columns to float for JSON serialization
X_example = X.head(5).copy()
for col in X_example.select_dtypes(include=['object']).columns:
    if X_example[col].apply(lambda x: isinstance(x, type(X_example[col].iloc[0])) and type(X_example[col].iloc[0]).__name__ == 'Decimal').any():
        X_example[col] = X_example[col].astype(float)


In [0]:
with mlflow.start_run(run_name=f"{best_model_name}_FINAL"):
    mlflow.sklearn.log_model(
        best_model,
        name="model",
        signature=signature,
        input_example=X_example,
        registered_model_name="district_priority_classifier"
    )
    mlflow.log_metric("final_cv_accuracy", comparison.loc[best_run_name, "accuracy"])
    mlflow.log_metric("final_high_recall", comparison.loc[best_run_name, "high_recall"])

    mlflow.log_metric("final_test_accuracy",test_accuracy)
    mlflow.log_metric("final_test_high_recall",test_high_recall)
    mlflow.log_metric("final_test_high_precision",test_high_precision)
    mlflow.log_metric("final_test_high_f1",test_high_f1)

print(f"Registered model: district_priority_classifier (based on {best_run_name})")

🔗 View Logged Model at: https://dbc-0ce523d1-9ca0.cloud.databricks.com/ml/experiments/2496663037125020/models/m-5554b82b424241178e473a7f60e19a4e?o=7474656442921141
2026/09/06 19:59:43 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "i.... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.validate_serving_input` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- child_anaemia_pct
- dropout_rate_pct
- full_immunization_pct
- health_insurance_pct
- household_sanitation_pct
- ...

Registered model 'district_priority_classif

Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '16' of model 'workspace.default.district_priority_classifier': https://dbc-0ce523d1-9ca0.cloud.databricks.com/explore/data/models/workspace/default/district_priority_classifier/version/16?o=7474656442921141


Registered model: district_priority_classifier (based on logistic_regression_tuned)


## STEP 41: Write predictions into the GOLD table
Adds `predicted_tier` directly onto `gold_district_priority`, so the index-based tier, the model's predicted tier, and whether they agree all live in ONE table — what Power BI, the AI brief, and n8n will all connect to.

In [0]:
# Generate ML predictions for all 602 districts
pdf["predicted_tier"] = best_model.predict(X_full)

# Compare rule-based DHEIS tier vs ML prediction
pdf["prediction_correct"] = (
    pdf["predicted_tier"] == pdf["priority_tier"]
)

print("Total districts:", len(pdf))
print(
    "Matching predictions:",
    pdf["prediction_correct"].sum()
)
print(
    "Different predictions:",
    (~pdf["prediction_correct"]).sum()
)



Total districts: 602
Matching predictions: 577
Different predictions: 25


In [0]:
final_gold_df = spark.createDataFrame(pdf)
final_gold_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_district_priority")

print("gold_district_priority updated. Final row count:", final_gold_df.count())

Final check — confirm the merged table looks right.

In [0]:
display(final_gold_df.select("district_name", "state_name", "priority_tier", "predicted_tier",
                               "prediction_correct").limit(10))

---
## Notebook Complete
